In [19]:
# 03_model_training.ipynb

# imports + config
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import os,sys

project_root = os.path.abspath("..")   # ← 根據你的資料夾結構調整
sys.path.insert(0, project_root)

print("Current working dir:", os.getcwd())
print("Project root added:", project_root)

Current working dir: /Users/mankin/Desktop/Anomaly_Detection_on_Graph/notebooks
Project root added: /Users/mankin/Desktop/Anomaly_Detection_on_Graph


In [20]:
import pandas as pd

features_path = "../data/raw/elliptic_txs_features.csv"
try:
    df = pd.read_csv(features_path, header=None)
    print("features.csv shape:", df.shape)  # 應為 (203769, 167)
    print("\n前 5 行，前 5 欄：\n", df.iloc[:5, :5])
    print("\n第 6 行，前 5 欄（確認 txId 格式）：\n", df.iloc[5:10, :5])
    print("\nColumn 0 (time step) sample:", df[0].head().tolist()) # 時間步驟
    print("Column 1 (txId) sample:", df[1].head().tolist())
except Exception as e:
    print("讀取失敗:", str(e))

features.csv shape: (203769, 167)

前 5 行，前 5 欄：
            0  1         2         3         4
0  230425980  1 -0.171469 -0.184668 -1.201369
1    5530458  1 -0.171484 -0.184668 -1.201369
2  232022460  1 -0.172107 -0.184668 -1.201369
3  232438397  1  0.163054  1.963790 -0.646376
4  230460314  1  1.011523 -0.081127 -1.201369

第 6 行，前 5 欄（確認 txId 格式）：
            0  1         2         3         4
5  230459870  1  0.961040 -0.081127 -1.201369
6  230333930  1 -0.171264 -0.184668 -1.201369
7  230595899  1 -0.171755 -0.184668 -1.201369
8  232013274  1 -0.123127 -0.184668 -1.201369
9  232029206  1 -0.005027  0.578941 -0.091383

Column 0 (time step) sample: [230425980, 5530458, 232022460, 232438397, 230460314]
Column 1 (txId) sample: [1, 1, 1, 1, 1]


In [21]:
edges_path = "../data/raw/elliptic_txs_edgelist.csv"
try:
    df_edges = pd.read_csv(edges_path, names=['txId1', 'txId2'])
    print("edgelist shape:", df_edges.shape)  # 應為 (234355, 2)
    print("\n前 5 行：\n", df_edges.head())
    print("\nSample txId1:", df_edges['txId1'].head().tolist())
except Exception as e:
    print("讀取 edgelist 失敗:", str(e))

edgelist shape: (234356, 2)

前 5 行：
        txId1      txId2
0      txId1      txId2
1  230425980    5530458
2  232022460  232438397
3  230460314  230459870
4  230333930  230595899

Sample txId1: ['txId1', '230425980', '232022460', '230460314', '230333930']


In [22]:
classes_path = "../data/raw/elliptic_txs_classes.csv"
try:
    df_classes = pd.read_csv(classes_path)
    print("classes shape:", df_classes.shape)  # (203769, 2)
    print("\n前 5 行：\n", df_classes.head())
except Exception as e:
    print("讀取 classes 失敗:", str(e))

classes shape: (203769, 2)

前 5 行：
         txId    class
0  230425980  unknown
1    5530458  unknown
2  232022460  unknown
3  232438397        2
4  230460314  unknown


In [23]:
# 確保每次訓練前都從頭開始處理資料，避免舊的 processed 檔案影響結果
processed_path = "../data/processed/elliptic_processed.pt"
if os.path.exists(processed_path):
    os.remove(processed_path)
    print("已成功刪除舊的 processed 檔案！下次執行 EllipticDataset 會重新處理")
else:
    print("找不到舊檔案，已經是乾淨狀態")

找不到舊檔案，已經是乾淨狀態


In [24]:
# 引入 config
from src.config import Config
cfg = Config()               # 建立 config 物件

# override 你想改的參數
cfg.hidden_dim = 128          # 64 -> 128
cfg.num_layers = 3            # 2 -> 3
cfg.lr = 0.005                # 0.01 -> 0.005   
cfg.epochs = 200              # 100 -> 200
cfg.patience = 30             # 20 -> 30
cfg.use_degree = True         # 明確設定
cfg.aggregator = "mean"       # 明確設定

In [25]:
# 定義 ImprovedGraphSAGE, cfg: hidden_dim, num_layers, dropout, aggregator
from torch_geometric.nn import SAGEConv     # SAGEConv 是 GraphSAGE 論文中提出的 GraphSAGE 卷積層，支援 mean、max、sum、lstm 等不同的鄰居聚合方式（aggregator）。

class ImprovedGraphSAGE(nn.Module):        # 定義一個新的 GraphSAGE 模型類別，命名為 ImprovedGraphSAGE，繼承自 nn.Module。
    def __init__(self, in_channels, hidden_channels=None, num_layers=None, dropout=0.2):  # 特徵維度 in_channels、隱藏層維度 hidden_channels、層數 num_layers 和 dropout 比例。
        super().__init__()  
        if hidden_channels is None:         
            hidden_channels = cfg.hidden_dim         # 如果沒有指定 hidden_channels，就從 config 讀取預設值。
        if num_layers is None:
            num_layers = cfg.num_layers              # 如果沒有指定 num_layers，就從 config 讀取預設值。
            
        self.convs = nn.ModuleList()                 # 建立一個 ModuleList，用來存放多層的 SAGEConv 層。
        self.convs.append(SAGEConv(in_channels, hidden_channels, aggr=cfg.aggregator)) # 第一層從 in_channels 映射到 hidden_channels，使用 config 中指定的 aggregator。
        for _ in range(num_layers - 2):              # ignore the first and last layer, add more hidden layers if num_layers > 2
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, aggr=cfg.aggregator))
            
        if num_layers > 1:                           # 最後一層從 hidden_channels 映射到 hidden_channels，因為最後還有一個線性層會從 hidden_channels 映射到 2。
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, aggr=cfg.aggregator))
            
        self.dropout = nn.Dropout(dropout)           # 定義 dropout 層，比例由參數 dropout 決定。
        self.lin = nn.Linear(hidden_channels, 2)     # 最後的線性層，將 hidden_channels 映射到 2，因為我們是二分類問題（正常 vs 非正常）。

    def forward(self, x, edge_index):                # 定義前向傳播函數，接受節點特徵 x [num_nodes, in_channels]和邊列表 edge_index [2, num_edges]。
        for conv in self.convs[:-1]:                 # 對前 num_layers-1 層的 SAGEConv 層進行迭代，對每一層先做卷積，再接 ReLU 激活函數，最後做 dropout。
            x = conv(x, edge_index).relu()           # 舊的 x 被覆蓋成新的 x。 根據 edge_index 找到每個節點的鄰居（neighbors）。
            x = self.dropout(x)
        if len(self.convs) > 0:
            x = self.convs[-1](x, edge_index)        # 最後一層的 SAGEConv 層不接 ReLU，直接輸出特徵，然後接 dropout 和線性層。
        x = self.dropout(x)                                 
        return self.lin(x)

/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
# 重新執行 notebook 中的資料載入 cell
# 載入資料（必須在 model 初始化之前執行）
from src.data import EllipticDataset   # 從 src.data 模組中引入 EllipticDataset 類別，這是我們之前定義的資料集類別，用來處理 Elliptic 資料集。

dataset = EllipticDataset(
    root=str(cfg.data_dir), 
    use_degree=cfg.use_degree,
    use_pagerank=cfg.use_pagerank
)
data = dataset[0]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x = data.x.to(device)
edge_index = data.edge_index.to(device)
y = data.y.to(device)

print(f"資料載入完成：{x.shape[0]} nodes, {edge_index.shape[1]} edges, {x.shape[1]} features")

Processing...


開始處理 Elliptic 資料集...
MARKER: process() method called - About to read classes
Nodes: 203769, Features dim: 165
Sample txIds (col 0): [230425980, 5530458, 232022460, 232438397, 230460314]
Sample time steps (col 1): [1, 1, 1, 1, 1]
edgelist 有效行數: 234355
Sample edges txId1 (first 5): [230425980, 232022460, 230460314, 230333930, 232013274]
MARKER: Starting edge matching loop
有效 directed edges: 234355
匹配成功率: 100.00% (234355/234355)
無向 edges: 468710
加入 degree → x dim: 166


/Users/mankin/Desktop/Anomaly_Detection_on_Graph/src/data.py:145: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  torch.save((self.data, self.slices), self.processed_paths[0])


處理完成！
最終: 203769 nodes, 468710 edges
資料載入完成：203769 nodes, 468710 edges, 166 features


Done!
/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([DataEdgeAttr])` to allowlist this global.
  out = fs.torch_load(path)


In [27]:
# 初始化 model
model = ImprovedGraphSAGE(
    in_channels=x.shape[1],
    hidden_channels=cfg.hidden_dim,
    num_layers=cfg.num_layers
).to(device)

print(model)

ImprovedGraphSAGE(
  (convs): ModuleList(
    (0): SAGEConv(166, 128, aggr=mean)
    (1-2): 2 x SAGEConv(128, 128, aggr=mean)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (lin): Linear(in_features=128, out_features=2, bias=True)
)


In [28]:
# 資料分割
# 使用 temporal split（推薦）或 fallback
time_step = getattr(data, 'time_steps', None)   # 注意：是 time_steps，不是 time_step

if time_step is not None:
    time_step = time_step.to(device)
    train_val_mask = (time_step <= 34) & (y != -1)
    test_mask      = (time_step > 34)  & (y != -1)
    
    labeled_indices = torch.where(train_val_mask)[0].cpu().numpy()
    train_idx, val_idx = train_test_split(
        labeled_indices, test_size=0.2,
        stratify=y[train_val_mask].cpu().numpy(), random_state=42
    )
    train_idx = torch.tensor(train_idx, device=device)
    val_idx   = torch.tensor(val_idx, device=device)
    test_idx  = torch.where(test_mask)[0]
    
    print("使用 Temporal Split（推薦）")
else:
    # 你的原本隨機分割（fallback）
    labeled_mask = y != -1
    labeled_indices = torch.where(labeled_mask)[0].cpu().numpy()
    train_val_idx, test_idx = train_test_split(labeled_indices, test_size=0.2, 
                                               stratify=y[labeled_mask].cpu().numpy(), random_state=42)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.2, 
                                          stratify=y[train_val_idx].cpu().numpy(), random_state=42)
    train_idx = torch.tensor(train_idx, device=device)
    val_idx = torch.tensor(val_idx, device=device)
    test_idx = torch.tensor(test_idx, device=device)
    
    print("使用 Random Split（fallback）")

print(f"訓練: {len(train_idx)} | 驗證: {len(val_idx)} | 測試: {len(test_idx)}")

使用 Temporal Split（推薦）
訓練: 23915 | 驗證: 5979 | 測試: 16670


In [29]:
# 訓練迴圈 + 評估
optimizer = Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

best_val_auc = 0.0
patience_counter = 0

for epoch in range(cfg.epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x, edge_index)
    loss = criterion(logits[train_idx], y[train_idx])
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(x, edge_index)
            val_probs = F.softmax(val_logits[val_idx], dim=1)[:, 1].cpu().numpy()
            val_auc = roc_auc_score(y[val_idx].cpu().numpy(), val_probs)
            
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Val AUC: {val_auc:.4f}")
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_counter = 0
                torch.save(model.state_dict(), "../saved_models/graphsage_best.pt")
            else:
                patience_counter += 1
                if patience_counter >= cfg.patience:
                    print("Early stopping!")
                    break

Epoch   0 | Loss: 0.9791 | Val AUC: 0.7236
Epoch  10 | Loss: 0.2225 | Val AUC: 0.9295
Epoch  20 | Loss: 0.1785 | Val AUC: 0.9532
Epoch  30 | Loss: 0.1223 | Val AUC: 0.9718
Epoch  40 | Loss: 0.0886 | Val AUC: 0.9806
Epoch  50 | Loss: 0.0673 | Val AUC: 0.9856
Epoch  60 | Loss: 0.0544 | Val AUC: 0.9886
Epoch  70 | Loss: 0.0396 | Val AUC: 0.9899
Epoch  80 | Loss: 0.0289 | Val AUC: 0.9909
Epoch  90 | Loss: 0.0237 | Val AUC: 0.9910
Epoch 100 | Loss: 0.0178 | Val AUC: 0.9913
Epoch 110 | Loss: 0.0150 | Val AUC: 0.9916
Epoch 120 | Loss: 0.0124 | Val AUC: 0.9912
Epoch 130 | Loss: 0.0094 | Val AUC: 0.9909
Epoch 140 | Loss: 0.0080 | Val AUC: 0.9909
Epoch 150 | Loss: 0.0075 | Val AUC: 0.9909
Epoch 160 | Loss: 0.0051 | Val AUC: 0.9909
Epoch 170 | Loss: 0.0061 | Val AUC: 0.9908
Epoch 180 | Loss: 0.0044 | Val AUC: 0.9903
Epoch 190 | Loss: 0.0036 | Val AUC: 0.9903


In [30]:
# 最終評估
@torch.no_grad()
def test(mask):
    model.eval()
    out = model(x, edge_index)
    pred = out.softmax(dim=1)[:, 1]
    y_true = y[mask].cpu().numpy()
    y_score = pred[mask].cpu().numpy()
    auc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    return auc, auprc

test_auc, test_auprc = test(test_idx)
print(f"Test AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f}")

Test AUC: 0.8923 | AUPRC: 0.6737
